In [1]:
import scipy.io
import numpy as np
import os
import glob
import torch
from scipy import signal 
from sklearn.model_selection import LeaveOneOut
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

In [2]:
def load_mat_file(filepath):
    """ Load .mat file and return Xtr, Ytr always; Xte, Yte only if they are not NaN """
    mat_data = scipy.io.loadmat(filepath)
    
    Xtr = mat_data['Xtrain']
    Ytr = mat_data['Ytrain']
    Xte = mat_data['Xtest']
    Yte = mat_data['Ytest']
    Yte_fb = mat_data['Ytest_fb']
    
    # Check if Xte and Yte are NaN or entirely NaN arrays
    if np.isnan(Xte).all() or np.isnan(Yte).all() or np.isnan(Yte_fb).all():
        # If all values in Xte or Yte are NaN, return only Xtr and Ytr
        return Xtr, Ytr
    else:
        # Otherwise, return Xtr, Ytr, Xte, Yte
        return Xtr, Ytr, Xte, Yte, Yte_fb

In [ ]:
# Euclidean Alignment (EA)
def euclidean_alignment(Xtarget, Xsource):
    """
    Align Xsource to the covariance structure of Xtarget using Euclidean Alignment.
    
    Parameters:
    Xtarget (ndarray): Target dataset of shape (n_trials, n_channels, n_samples).
    Xsource (ndarray): Source dataset of shape (n_trials, n_channels, n_samples).
    
    Returns:
    Xadapted (ndarray): Adapted source dataset aligned to target, same shape as Xsource.
    """
    
    # Get the number of trials, channels, and samples
    n_trials, n_channels, n_samples = Xsource.shape
    
    # Step 1: Compute the average covariance matrix of all Xtarget trials
    cov_target = np.zeros((n_channels, n_channels))
    
    for trial in range(Xtarget.shape[0]):
        cov_target += np.cov(Xtarget[trial])
        
    cov_target /= Xtarget.shape[0]
    
    # Step 2: Compute the inverse square root of the average covariance matrix
    eigvals, eigvecs = np.linalg.eigh(cov_target)
    R_inv_sqrt = eigvecs @ np.diag(1.0 / np.sqrt(eigvals)) @ eigvecs.T
    
    # Step 3: Apply this transformation to each trial in Xsource
    Xadapted = np.zeros_like(Xsource)
    
    for trial in range(n_trials):
        Xadapted[trial] = R_inv_sqrt @ Xsource[trial]
    
    return Xadapted




